In [3]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")


In [4]:
df = pd.read_csv("medium_post_titles.csv", nrows=10000)
df.head()

,category,title,subtitle,subtitle_truncated_flag
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False
3,equality,"""Call me Nat Love"" :Black Cowboys and the Fron...",NaN,False
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False


In [5]:
df['subtitle_truncated_flag'].value_counts()

subtitle_truncated_flag
False    6318
True     3682
Name: count, dtype: int64

In [6]:
df.isnull().sum()

category                     0
title                        0
subtitle                   107
subtitle_truncated_flag      0
dtype: int64

In [7]:
df = df.dropna()

In [8]:
df.isnull().sum()

category                   0
title                      0
subtitle                   0
subtitle_truncated_flag    0
dtype: int64

In [9]:
df = df.loc[~df['subtitle_truncated_flag']]

In [10]:
df['subtitle_truncated_flag'].value_counts()

subtitle_truncated_flag
False    6211
Name: count, dtype: int64

In [11]:
df.shape

(6211, 4)

In [12]:
# cleaning data

df['title_extened'] = df['title'] + df['subtitle']
df['title_extened']

0       "21 Conversations" - A fun (and easy) game for...
1       "Biblical Porn" at Mars HillAuthor and UW lect...
2       "CISGENDER?! Is That A Disease?!"Or, a primer ...
4       "Can I Train my Model on Your Computer?"How we...
5       "Cypherpunks and Wall Street": The Security To...
                              ...                        
9994    America Lets Too Much Young Talent Go to Waste...
9996    America Loves the Idea of Family Farms. That’s...
9997    America May Need to Adopt China’s Weapons to W...
9998    America May Outsmart China in 5G With AI and B...
9999    America Needs Bernie SandersIn this crucial mo...
Name: title_extened, Length: 6211, dtype: object

In [13]:
df['category'].nunique()

93

In [14]:
df.shape

(6211, 5)

In [ ]:
# prepare data for upsert into pinecone
#init pinecone

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=pinecone_api_key)

In [46]:
pc.create_index(
    name="medium-posts",
    dimension=384,
    metric="cosine",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

IndexModel(name='medium-posts', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://medium-posts-v29tsf3.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=384, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [47]:
pc.list_indexes()

IndexList([<name='medium-posts', dim=384, ready=True>])

In [24]:
from sentence_transformers import SentenceTransformer
import torch


In [25]:
# embedding model

model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\acer\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [28]:
df['values'] = df['title_extened'].apply(lambda x: model.encode(x).tolist())

In [30]:
# having vec ids for the dataframe

df['ids'] = df.reset_index(drop=True).index.astype(str)

In [33]:
df['metadata'] = df.apply(lambda x: {"title": x['title'], "subtitle": x['subtitle'], "category": x['category']}, axis=1)

In [35]:
df.drop(['title', 'subtitle', 'subtitle_truncated_flag', 'title_extened', 'category'], axis=1, inplace=True)

In [40]:
df_c = df[['ids', 'values', 'metadata']].copy()

In [48]:
idx = pc.index("medium-posts")

In [49]:
vectors = []

for _, row in df_c.iterrows():
    vectors.append({
        "id": row["ids"],
        "values": row["values"],
        "metadata": row["metadata"]
    })

batch_size = 100

for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i + batch_size]
    idx.upsert(vectors=batch)

In [56]:
query_embedding = model.encode("where is that my candy?").tolist()

result = idx.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

for res in result['matches']:
    print(f"ID: {res['id']}, Score: {res['score']}, title: {res['metadata'].get('title', 'N/A')}, subtitle: {res['metadata'].get('subtitle', 'N/A')}, category: {res['metadata'].get('category', 'N/A')}")

ID: 4817, Score: 0.370415688, title: A six year old bar-fly., subtitle: How I lost my tooth in a bar., category: family
ID: 3598, Score: 0.352658272, title: A Mixed Bag at the Farmers’ Market, subtitle: This week in my ongoing quest to avoid plastic I visited the farmers’ market, where I got more – and less – than I bargained for., category: food
ID: 2761, Score: 0.349910766, title: A Child is Like a Box of Chocolates, subtitle: You never know what you’re going to get., category: parenting
ID: 5273, Score: 0.326788902, title: Across The Pond, subtitle: Romance in the city., category: fiction
ID: 4303, Score: 0.316690445, title: A Toothpick at Every Meal, subtitle: A family tree. Oklahoma, 2006., category: comics
